# Parameter exploration

Sensitivity of the broken-down-lorry bottleneck to **driver heterogeneity** and of the moving-lorry scenario to **lorry density**. Short runs keep this quick (~2-3 min total. `exit_rate_veh_h` counts vehicles leaving the road per hour of simulated time, so it includes the ~160 s warm-up while the empty road fills: compare runs relatively, not as absolute capacity).

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

from motorway import Simulation, load_config
from motorway.analytics import summary


def run(config_path=None, **overrides):
    sim = Simulation(load_config(config_path, **overrides))
    df = sim.run()
    s = summary(df)
    s['exit_rate_veh_h'] = sim.exited / sim.t * 3600
    return s

## Driver heterogeneity
Share of aggressive drivers (short headway, low politeness) vs. cautious ones, at the default bottleneck.

In [ ]:
rows = []
for aggressive in [0.0, 0.2, 0.4, 0.6]:
    mix = {'aggressive': aggressive, 'normal': 0.6, 'cautious': 0.4 - aggressive * 2 / 3}
    rows.append({'aggressive_share': aggressive, **run('../config.yaml', duration=240, profile_mix=mix)})
hetero = pd.DataFrame(rows)
hetero

## Lorry density
Share of slow-lane arrivals that are lorries, in the moving-bottleneck scenario.

In [ ]:
rows = []
for frac in [0.0, 0.1, 0.2, 0.3]:
    rows.append({'lorry_fraction': frac, **run('../config_moving_lorry.yaml', duration=240, lorry_fraction=frac)})
lorries = pd.DataFrame(rows)
lorries

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hetero.plot(x='aggressive_share', y=['mean_speed_kmh'], marker='o', ax=axes[0], title='Heterogeneity')
ax2 = axes[0].twinx(); hetero.plot(x='aggressive_share', y='exit_rate_veh_h', marker='s', color='C1', ax=ax2, legend=False)
lorries.plot(x='lorry_fraction', y=['mean_speed_kmh'], marker='o', ax=axes[1], title='Lorry density')
ax3 = axes[1].twinx(); lorries.plot(x='lorry_fraction', y='exit_rate_veh_h', marker='s', color='C1', ax=ax3, legend=False)
for ax in (ax2, ax3): ax.set_ylabel('exit rate (veh/h)')
fig.tight_layout()